In [2]:
from Adversarial_training import get_adv_data,train_loop,test_loop
from ATKMethods import *
from scipy.io import loadmat,savemat
from sklearn.model_selection import train_test_split
import numpy as np
import utils.gestureDataLoader as gestureDataLoader
import tensorflow as tf
import os
import utils.Config as Config
import random
random.seed(42)
config = Config.getconfig( )
config.source = 'lab_276'
tf.config.experimental_run_functions_eagerly(True)
assert config.source != None, 'source should not be None'
train_data, test_data, train_label, test_label = gestureDataLoader.getData(
        config, 'signfi'
        )

X_train, X_test, y_train, y_test = train_test_split( train_data, train_label, test_size=0.1, random_state=42)
batch_size = config.batch_size
# Prepare the training dataset.
train_dataset = tf.data.Dataset.from_tensor_slices((X_train, y_train))
train_dataset = train_dataset.shuffle(buffer_size=1024).batch(batch_size)

val_dataset = tf.data.Dataset.from_tensor_slices((X_test, y_test))
val_dataset = val_dataset.batch(batch_size)

test_dataset = tf.data.Dataset.from_tensor_slices((test_data, test_label))
test_dataset = test_dataset.batch(batch_size)

# extract model training information
def extract_info(filename):
    filename = filename.replace('.h5','')
    if 'robust_adv_training' in filename:
        adv_training_method = filename.split('robust_adv_training_')[1].split('_')[0]
        psr = float(filename.split('psr_')[1].split('_')[0])
        model_name = filename.split('_')[6]
        n_iters = 'None' if 'fgsm' in filename else int(filename.split('niter_')[1].split('_')[0])
        return adv_training_method,psr,model_name,n_iters
    else:
        model_name = filename.split('_')[1]
        return 'Normal',None,model_name,None

Loading data................


## Build Model

In [3]:
config.DNN_name = 'defult'
config.epoch = 1000
# 0.5e-3
# 0.075e-3
psr = 1e-3
method = 'pgd'
n_iter = 8
model_name = f'robust_adv_training_{method}_psr_{psr}_{config.DNN_name}_{config.source}_niter_{n_iter}_new.h5'
config.model_path['adv_robust_model_path'] = 'SavedModel/Adversarial_robust_model' 
config.model_path['adv_robust_model_path'] = os.path.join( config.model_path['adv_robust_model_path'], model_name)
net = AlexNetTF( config )
model = net.buildModel( choice = config.DNN_name)

# print(f'The model existance? ',os.path.exists(config.model_path['adv_robust_model_path']))
if os.path.exists(config.model_path['adv_robust_model_path']):
    model.load_weights(config.model_path['adv_robust_model_path'])
    print(f'The model existance? ',os.path.exists(config.model_path['adv_robust_model_path']))
    print('load the model',config.model_path['adv_robust_model_path'])
else:
    print('Model not exist, create new model: \n',config.model_path['adv_robust_model_path'])

The model existance?  True
load the model SavedModel/Adversarial_robust_model/robust_adv_training_pgd_psr_0.001_defult_home_276_niter_8_new.h5


## Adversarial Training

In [ ]:
# for current_model in ['defult',]:
for psr in [0.75e-3, ]:
    for n_iter in [4,]:
        config.DNN_name = 'defult'
        config.epoch = 1000
        method = 'pgd'
        model_name = f'robust_adv_training_{method}_psr_{psr}_{config.DNN_name}_{config.source}_niter_{n_iter}_new.h5'
        config.model_path['adv_robust_model_path'] = os.path.join( 'SavedModel/Adversarial_robust_model' , model_name)
        net = AlexNetTF( config )
        model = net.buildModel( choice = config.DNN_name)
        if os.path.exists(config.model_path['adv_robust_model_path']):
            model.load_weights(config.model_path['adv_robust_model_path'])
            print(f'The model existance? ',os.path.exists(config.model_path['adv_robust_model_path']))
            print('load the model',config.model_path['adv_robust_model_path'])
        else:
            print('Model not exist, create new model: \n',config.model_path['adv_robust_model_path'])
        config.lr = 1e-5
        model = train_loop(config,model,train_dataset,val_dataset,psr,method,n_iter = n_iter)
        print(f"The accuracy of model {config.model_path['adv_robust_model_path']} is",f'{test_loop(None,None,model,test_dataset,None).numpy()*100}%',sep= '\t')

In [ ]:
for test_list in os.listdir('SavedModel/Adversarial_robust_model'):
    if 'pgd' in test_list and 'defult' in test_list:
        if test_list.split('_')[-2] == '4':
            config.DNN_name = test_list.split('_')[6]
            net = AlexNetTF( config )
            model = net.buildModel( choice = config.DNN_name)
            model.load_weights('SavedModel/Adversarial_robust_model'+'/'+test_list)
            print(f"The accuracy of model {test_list} is",f'{test_loop(None,None,model,test_dataset,None).numpy()*100:.2f}%',sep= '\t')
        

## White Box Attack

### Evaluate on adversarial samples

In [ ]:
# model_path_list = os.listdir('SavedModel/Adversarial_robust_model')
# model_path_list = []
# for path in os.listdir('SavedModel/Adversarial_robust_model'):
#     if 'L_inf' in path:
#         continue
#     model_path_list.append(path)
# model_path_dict = {}
# for path in model_path_list:
#     adv_training_method,psr,model_name,n_iters = extract_info(path)
#     model_path_dict[f'{model_name}_{adv_training_method}_{n_iters}_{psr}'] = path
# mat_path = 'resultsMat/adversarial_trainin_alex_resnet.mat'
# psr_all = [ 0.0, 6.250e-05, 1.250e-04, 1.875e-04 ,2.500e-04 ,3.125e-04 ,3.750e-04, 4.375e-04, 5.000e-04, 1.000e-03 ,1.500e-03 ,3.000e-03, 4.000e-03, 5.000e-03]
mat_path = 'resultsMat/adversarial_trainin_alex_resnet_fineG_5e6to5e2.mat'
psr_all = np.exp(np.linspace(np.log(5e-6),np.log(50e-3),100))
# mat_path = 'resultsMat/adversarial_trainin_alex_resnet_fineG_5e6to5e2_linear.mat'
# psr_all = np.linspace(0.005e-3,50e-3,100)
if os.path.exists(mat_path):
    print('mat file exist')
    acc_all_adv = loadmat(mat_path,squeeze_me=True)
    psr_all = acc_all_adv['psr']
    print('loaded mat file')
else:
    print('mat file not exist, build a new one')
    acc_all_adv = {}
    psr_all = np.exp(np.linspace(np.log(5e-6),np.log(50e-3),100))
for ATK in ['fgsm','pgd_4','pgd_8','pgd_10','pgd_16','noise']:
    if ATK == 'fgsm':
        ATK_method = 'fgsm'
        iter = None
    elif ATK == 'noise':
        ATK_method = 'noise'
        iter = None
    else:
        ATK_method = ATK.split('_')[0]
        iter = int(ATK.split('_')[1])
    for model_path in os.listdir('SavedModel/Adversarial_robust_model'):
        if '.h5' not in model_path:
            continue
        acc_buf = []
        adv_training_method,psr,model_name,train_iters = extract_info(model_path)
        config.DNN_name = model_name
        out_key = f'{model_name}_{adv_training_method}_{train_iters}_{psr}_defense_against_{ATK_method}_{iter}'
        if out_key in acc_all_adv.keys():
            print(f'{out_key} already exist')
            continue
        net = AlexNetTF( config )
        model = net.buildModel( choice = config.DNN_name)
        model.load_weights(os.path.join('SavedModel/Adversarial_robust_model',model_path))
        for psr_current in psr_all:
            test_acc = test_loop(config,psr_current,model,test_dataset,ATK_method,n_iter = iter)
            acc_buf.append( test_acc )
        print(f'Adversarial training model: {model_name} method: {adv_training_method} iters: {train_iters} psr: {psr} defense against {ATK_method} {iter} \n', )

        acc_all_adv.update({ 
                            'psr':psr_all,
                            out_key: np.asarray(acc_buf)
                            })
    savemat(mat_path,acc_all_adv)

### Evaluate on clean samples

In [3]:
# for path in os.listdir('SavedModel/Adversarial_robust_model'):
#     if '.h5' not in path:
#         continue
#     if 'Normal' in path:
#         model_name = path.split('_')[1].split('.')[0]
#     else:
#         _,_,model_name,_ = extract_info(path)

#     if 'home' in path:
path = 'robust_adv_training_pgd_psr_0.00075_defult_lab_276_niter_16_new.h5'  
net = AlexNetTF( config )
model = net.buildModel( choice = 'defult')
model.load_weights(os.path.join('SavedModel/Adversarial_robust_model',path))
print(f"The accuracy of model {path} is",test_loop(None,None,model,test_dataset,None).numpy(),sep= '\t')

2022-12-06 14:42:10.908788: I tensorflow/stream_executor/cuda/cuda_dnn.cc:366] Loaded cuDNN version 8201
2022-12-06 14:42:12.159884: I tensorflow/stream_executor/cuda/cuda_blas.cc:1774] TensorFloat-32 will be used for the matrix multiplication. This will only be logged once.


The accuracy of model robust_adv_training_pgd_psr_0.00075_defult_lab_276_niter_16_new.h5 is	0.0027173914


### Evaluate on deepfool

In [ ]:
pert_path = f'perturbation/deepfool/{path}_df.mat'.replace('.h5','')

In [ ]:

config.DNN_name = 'resnet'
config.epoch = 1000
print(config.N_classes)
np.set_printoptions(precision=3)

out_acc = {}
for path in os.listdir('SavedModel/Adversarial_robust_model'):
    if '.h5' not in path:
        continue
    adv_training_method,psr_model,model_name,train_iters = extract_info(path)
    net = AlexNetTF( config )
    model = net.buildModel( choice = model_name)
    model.load_weights(f'SavedModel/Adversarial_robust_model/{path}')
    
    pert_path = f'perturbation/deepfool/{path}_df.mat'.replace('.h5','')
    if not os.path.exists(pert_path):
        gen_adv_data(test_data, test_label, model, 'deepfool',psr,model_path = path.replace('.h5',''))
    df_data = loadmat(pert_path,squeeze_me=True)
    out_key = f'{model_name}_{adv_training_method}_{train_iters}_{psr_model}_defense_against_deepfool'
    acc = []
    if os.path.exists('resultsMat/robust_model_against_deepfool.mat'):
        out_acc = loadmat('resultsMat/robust_model_against_deepfool.mat',squeeze_me=True)
    if out_key in out_acc.keys():
        print(path)
        print(out_acc[out_key])
        continue
    for psr in loadmat('resultsMat/adversarial_trainin_alex_resnet.mat',squeeze_me=True)['psr']:
        test_dataset = tf.data.Dataset.from_tensor_slices((df_data['x']+l2_limiter(psr,df_data['delta'],df_data['x']), df_data['y']))
        test_dataset = test_dataset.batch(64)
        acc.append(np.array(test_loop(None,None,model,test_dataset,None)))
    print(path)
    acc = np.asarray(acc)
    print('The accuracy is ',acc,)
    out_acc.update({out_key:acc})
    savemat('resultsMat/robust_model_against_deepfool.mat',out_acc)

In [ ]:
import matplotlib.pyplot as plt
from utils.TOOLS import scaleDeepfool
from ATKMethods import *
df_data = loadmat('perturbation/deepfool/df_perturbation.mat',squeeze_me=True)

# df_data['delta'] = scaleDeepfool(0.0005,df_data['x'],df_data['delta'])
df_data['delta'] = l2_limiter(0.0005,df_data['delta'],df_data['x'])
plt.plot(df_data['x'][0,:,0,0])
adv_data = df_data['x'][0,:,0,0] + df_data['delta'][0,:,0,0]

plt.plot(adv_data)

In [ ]:
from ATKMethods import *
print(df_data['x'].shape)
print(df_data['delta'].shape)
# delta = scaleDeepfool(0.0005,df_data['x'],df_data['delta'])
for x,delta in zip(df_data['x'],df_data['delta']):
    x = np.expand_dims(delta,axis=0) 
    delta = np.expand_dims(delta,axis=0)
    delta = scaleDeepfool(0.0005,x,delta)
    print(compute_psr(delta - delta.mean(),x))

## Black Box Attack (UAP)

In [2]:
import h5py
from utils.Universal_pert import universal_perturbation
# from utils.TOOLS import genereate_UAP
from utils.TOOLS import scaleDeepfool
import copy
UAP_save_folder = 'perturbation/UAP_AT_model/'
def save_UAP(UAP_name,UAP_data):
    with h5py.File( UAP_save_folder + UAP_name, 'w' ) as hdf:
        hdf.create_dataset( 'universal_perturbation', data = UAP_data )
def genereate_UAP(dataset,model,config):
    '''
    :param dataset: the dataset to loop over
    :param model_path: the attack model path
    :return: the UAP
    '''
    f = tf.keras.Model( model.input, model.layers[ -2 ].output )
    if f.output_shape[ 1 ] != config.N_classes:
        raise Exception(
                f'The output of the feed forward function is wrong, the output should be {config.N_classes}, '
                f'but it is {f.output_shape[ 1 ]}'
                )
    UAP = universal_perturbation( dataset = dataset, f = f, overshoot = 0.002 )
    return UAP
config.DNN_name = 'defult'
def UAPTest(
        X,
        y,
        victim_model = None,
        psr_range = None,
        **UAP_file_names):
    acc_all = {}
    # psr_range = np.arange( 0.000, 0.007, 0.0005 )
    if psr_range is None:
        psr_range = [0,0.0035,0.004]
    # psr_range = np.arange(0,0.0041,0.0005)
    acc_all['Guassian_noise'] = []
    for name in UAP_file_names.keys():
        
        # desc = f'{name}'# victim: {victim_name}'
        acc_all[name] = []
        
        with h5py.File(UAP_file_names[name],'r') as f:
            UAP_data = np.asarray(list( f[ 'universal_perturbation' ] ))
        for psr in tqdm(psr_range,position = 0):
            # Perturbation calibration
            scaled_uni_per = scaleDeepfool(psr = psr,x = X, perturbation = UAP_data)
            adv_data = X + scaled_uni_per - scaled_uni_per.mean()
            # Testing
            test_ds = tf.data.Dataset.from_tensor_slices((adv_data, y))
            test_ds = test_ds.shuffle(buffer_size=1024).batch(config.batch_size)
            acc = test_loop(None,psr,victim_model,test_ds,method = None)
            acc_all[ name ].append(acc.numpy())
    return acc_all
def loop_filter(path,bool_mode = 'and',*args):
    '''
    *args list: [term, 'in' or 'not in' or '==' or '!=']
    '''
    bool_list = []
    for i in args:
        method = i[1]
        if method == 'in':
            bool_list.append( i in path)
        elif method == 'not in':
            bool_list.append( i not in path)
        elif method == '==':
            bool_list.append( i == path)
        elif method == '!=':
            bool_list.append( i != path)
        else:
            raise Exception('The method is not supported')
        if bool_mode == 'and':
            return all(bool_list)
        elif bool_mode == 'or':
            return any(bool_list)
        else:
            raise Exception('The bool_mode is not supported')
        

### Generating UAP

In [ ]:
data          = copy.deepcopy( np.concatenate( (X_train,X_test), axis = 0 ) )
test_label    = copy.deepcopy( np.concatenate( (y_train,y_test), axis = 0 ) )
seed_container    = [ 2,3,4,5, 6, 7, 8, 9, 10,42 ]
for model_path in os.listdir('SavedModel/Adversarial_robust_model'):
    config.model_path['adv_robust_model_path'] = 'SavedModel/Adversarial_robust_model/' + model_path
    if '.h5' not in model_path or 'resnet' in model_path:
        continue
    net = AlexNetTF( config )
    model = net.buildModel( choice = 'defult')
    model.load_weights(config.model_path['adv_robust_model_path'])
    UAP_data     = genereate_UAP( dataset = data, model = model, config = config)
    UAP_name        = 'UAP_' + config.model_path['adv_robust_model_path'].split( '/' )[ -1 ]
    path            = os.path.join( config.pert_Mat_Root, UAP_name )
    save_UAP(UAP_name,UAP_data)

### Testing UAP

In [ ]:
path = 'resultsMat/AP_cross_model_0to0.02.mat'
victim_model_dir = 'SavedModel/Adversarial_robust_model/'
UAP_dir = 'perturbation/UAP_AT_model/'
psr_range = np.linspace(0,2e-2,21)

test_result = loadmat(path,squeeze_me=True) if os.path.exists(path) else {}
i = 0
for model_path in os.listdir(victim_model_dir):
    for UAP_path in os.listdir(UAP_dir):
        if '.h5' not in model_path or 'resnet' in model_path:
            continue
        if '.h5' not in UAP_path or 'resnet' in UAP_path:
            continue
        if 'home' in UAP_path or 'home' in model_path:
            continue
        # UAP_path = 'perturbation/UAP_AT_model/UAP_' + model_path
        # load model
        # net = AlexNetTF( config )
        # model = net.buildModel( choice = 'defult')
        # model.load_weights(victim_model_dir + model_path)
        UAP_info = {
            model_path: UAP_dir + UAP_path
        }
        if 'Normal' in model_path or 'Normal' in UAP_path:
            try:
                psr = model_path.split('_')[5]
                model_name = model_path.split('_')[6]
                at_method = model_path.split('_')[3]
                n_iter = model_path.split('_')[-2]
                vic_name = f'{model_name}_{at_method}_{n_iter}_{psr}'
            except:
                vic_name = 'Normal'
            try:
                UAP_psr = UAP_path.split('_')[6]
                UAP_model_name = UAP_path.split('_')[7]
                UAP_at_method = UAP_path.split('_')[4]
                UAP_n_iter = UAP_path.split('_')[-2]
                atk_name = f'{UAP_model_name}_{UAP_at_method}_{UAP_n_iter}_{UAP_psr}'
            except:
                atk_name = 'Normal'
            m_result_str = f'{atk_name}_UAPName_{vic_name}'
        elif 'robust' in model_path:
            psr = model_path.split('_')[5]
            model_name = model_path.split('_')[6]
            at_method = model_path.split('_')[3]
            n_iter = model_path.split('_')[-2]

            UAP_psr = UAP_path.split('_')[6]
            UAP_model_name = UAP_path.split('_')[7]
            UAP_at_method = UAP_path.split('_')[4]
            UAP_n_iter = UAP_path.split('_')[-2]
            m_result_str = f'{model_name}_{at_method}_{n_iter}_{psr}_UAPName_{UAP_model_name}_{UAP_at_method}_{UAP_n_iter}_{UAP_psr}'
        if m_result_str in test_result.keys():
            continue
        print(m_result_str)
        print(model_path)
        print(UAP_path)
        print('=====================')

        # acc_all = UAPTest(
        #                                 X_test,y_test,
        #                                 model,
        #                                 psr_range = psr_range,
        #                                 **UAP_info
        #                             )
        # acc = acc_all[list(acc_all.keys())[1]]
        # test_result[m_result_str] = acc
        # print(m_result_str,acc)
# savemat(path,test_result)

### Show Results

In [1]:
'defult_fgsm_None_0.00075_UAPName_defult_fgsm_None_0.003'
from scipy.io import loadmat, savemat
import numpy as np
psr_rec = loadmat('resultsMat/UAP_cross_model_0to0.02.mat',squeeze_me=True)
# print(psr_rec.keys())
def extractor( name ):
    model_name = name.split('_')[0]
    at_method = name.split('_')[1]
    n_iter = name.split('_')[2]
    psr = float(name.split('_')[3])
    UAP_model_name = name.split('_')[5]
    UAP_at_method = name.split('_')[6]
    UAP_n_iter = name.split('_')[7]
    UAP_psr = float(name.split('_')[8])
    return model_name,at_method,n_iter,psr,UAP_model_name,UAP_at_method,UAP_n_iter,UAP_psr
for key in psr_rec.keys():
    if key.startswith('__'):
        continue
    try:
        model_name,at_method,n_iter,psr,UAP_model_name,UAP_at_method,UAP_n_iter,UAP_psr = extractor(key)
        # print('a')
    except:
        print(key)
    if 'niter' in key:
        continue
    if at_method != 'fgsm':
        continue
    if UAP_at_method != 'fgsm':
        continue
    if 0.001 != psr:
        continue
    np.set_printoptions(precision=3)
    print(key,np.array(psr_rec[key]))

defult_fgsm_None_0.001_UAPName_defult_fgsm_None_0.001 [0.946 0.878 0.584 0.278 0.165 0.115 0.086 0.072 0.061 0.054 0.05  0.045
 0.038 0.032 0.029 0.023 0.018 0.016 0.016 0.014 0.009]
defult_fgsm_None_0.001_UAPName_defult_fgsm_None_0.005 [0.946 0.946 0.946 0.946 0.946 0.946 0.946 0.946 0.943 0.943 0.941 0.941
 0.939 0.939 0.93  0.928 0.919 0.896 0.882 0.862 0.851]
defult_fgsm_None_0.001_UAPName_defult_fgsm_None_5e-05 [0.946 0.946 0.943 0.939 0.937 0.93  0.914 0.905 0.882 0.869 0.848 0.842
 0.826 0.803 0.785 0.774 0.758 0.747 0.719 0.699 0.69 ]
defult_fgsm_None_0.001_UAPName_defult_fgsm_None_0.003 [0.946 0.946 0.946 0.943 0.943 0.943 0.941 0.939 0.939 0.939 0.928 0.921
 0.919 0.91  0.91  0.905 0.896 0.882 0.876 0.869 0.864]
defult_fgsm_None_0.001_UAPName_defult_fgsm_None_0.00025 [0.946 0.946 0.946 0.943 0.93  0.921 0.91  0.885 0.862 0.817 0.747 0.69
 0.627 0.581 0.534 0.482 0.439 0.389 0.362 0.326 0.274]
defult_fgsm_None_0.001_UAPName_defult_fgsm_None_0.00075 [0.946 0.937 0.894 0.742 0.6

In [79]:
for key in sorted(test_result.keys()):
    if 'pgd_4' not in key and 'fgsm_None_0.001' not in key:
        continue
    out = np.array(test_result[key])
    
    out = (out[0] - out)/out[0]
    np.set_printoptions(precision=3)
    print(key,'',out, sep='\t')

defult_fgsm_None_0.001		[0.    0.474 0.856 0.923 0.943 0.952 0.969 0.981]
defult_pgd_4_0.001		[0.    0.076 0.388 0.58  0.707 0.778 0.818 0.848]
